<a href="https://colab.research.google.com/github/cemgurbey/plastic-water-segmentation/blob/main/plastic_water_segmentation_cocosynth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌊 Plastic Residue Detection on Water Bodies using CocoSynth & Mask R-CNN

---

### Project Objective
Detect and segment plastic residue, floating litter, and aquatic debris on water bodies (rivers, lakes, harbors, coastlines, and open ocean) using instance segmentation.

### Pipeline Overview
1. **Synthetic Data Synthesis (CocoSynth)**: Composite transparent plastic item cutouts onto aquatic background scenes using realistic aquatic transformations (rotation, scaling, color jitter, water-surface alpha blending, reflection simulation).
2. **COCO Annotation Formatting**: Automatically extract polygon contours, bounding boxes, area, and categories to generate standard `coco_instances.json` annotations.
3. **Procedural Demo Generator**: Includes built-in procedural generators for water ripple textures and plastic cutouts so you can test the entire pipeline end-to-end immediately before uploading real photos.
4. **Mask R-CNN Training (TensorFlow 2.x & Keras)**: Uses `akTwelve/Mask_RCNN` (TensorFlow 2.x port) with transfer learning from pretrained COCO weights.
5. **Real-World Inference & Aquatic Contamination Index**: Detect plastic residue on unseen test water images and compute the percentage of water surface contaminated by plastic debris.

## 🛠️ Section 1: Setup & Dependencies
Here we verify hardware acceleration (GPU), install necessary libraries, and clone the TensorFlow 2.x compatible `akTwelve/Mask_RCNN` repository.

In [ ]:
import os
import sys
import json
import random
import math
import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageEnhance, ImageFilter, ImageDraw
import cv2
from shapely.geometry import Polygon, MultiPolygon
from tqdm import tqdm

# Check TensorFlow and GPU
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"Available GPUs: {len(gpus)}")
for gpu in gpus:
    print(f"  - {gpu}")

# Clone akTwelve/Mask_RCNN (TensorFlow 2.x compatible fork of Matterport Mask_RCNN)
MASK_RCNN_DIR = os.path.abspath("./Mask_RCNN")
if not os.path.exists(MASK_RCNN_DIR):
    print("Cloning akTwelve/Mask_RCNN repository...")
    !git clone https://github.com/akTwelve/Mask_RCNN.git {MASK_RCNN_DIR}
else:
    print(f"Mask_RCNN repository found at: {MASK_RCNN_DIR}")

# Add Mask_RCNN directory to sys.path
if MASK_RCNN_DIR not in sys.path:
    sys.path.append(MASK_RCNN_DIR)

from mrcnn.config import Config
import mrcnn.utils as utils
from mrcnn import visualize
import mrcnn.model as modellib
print("mrcnn modules successfully imported!")

## 📁 Section 2: Directory Layout
Organize directories for:
- `data/foregrounds/`: Plastic item cutouts with transparent backgrounds (PNG with alpha channel), organized by category.
- `data/backgrounds/`: Water body photos (lakes, rivers, ocean, ponds, canals).
- `dataset/train/` & `dataset/val/`: Generated synthetic images and COCO JSON annotation files.
- `test_images/`: Real-world photos of water bodies for final inference.
- `logs/`: Checkpoints and training weights.

In [ ]:
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
FOREGROUNDS_DIR = DATA_DIR / "foregrounds"
BACKGROUNDS_DIR = DATA_DIR / "backgrounds"
DATASET_DIR = BASE_DIR / "dataset"
TEST_IMAGES_DIR = BASE_DIR / "test_images"
MODEL_DIR = BASE_DIR / "logs"

# Define target plastic categories
CATEGORIES = ["plastic_bottle", "plastic_bag", "plastic_debris"]

# Create category subdirectories under foregrounds
for cat in CATEGORIES:
    (FOREGROUNDS_DIR / cat).mkdir(parents=True, exist_ok=True)

BACKGROUNDS_DIR.mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "train" / "images").mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "val" / "images").mkdir(parents=True, exist_ok=True)
TEST_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directories initialized:")
print(f"  - Foregrounds: {FOREGROUNDS_DIR}")
print(f"  - Backgrounds: {BACKGROUNDS_DIR}")
print(f"  - Dataset:     {DATASET_DIR}")
print(f"  - Model Logs:  {MODEL_DIR}")

## 🎨 Section 3: Procedural Demo Asset Generator
If you haven't uploaded images yet, this cell creates procedural water surface textures and transparent plastic shapes so you can test the entire pipeline immediately.

> **Note:** When you have your own images:
> - Place transparent PNG cutouts of plastic items into `data/foregrounds/<category>/`
> - Place water surface background images into `data/backgrounds/`

In [ ]:
def create_demo_assets():
    """Generates procedural water textures and transparent plastic cutouts if folders are empty."""
    existing_bgs = list(BACKGROUNDS_DIR.glob("*.jpg")) + list(BACKGROUNDS_DIR.glob("*.png"))
    if len(existing_bgs) > 0:
        print(f"Found {len(existing_bgs)} existing background images. Skipping demo generator.")
        return

    print("No background images detected. Generating procedural water textures...")
    water_palettes = [
        # Deep ocean blue
        ((15, 60, 115), (35, 115, 175), (100, 195, 235)),
        # River / Turbid green-brown
        ((30, 65, 50), (60, 105, 75), (120, 155, 105)),
        # Coastal aqua-teal
        ((10, 80, 95), (25, 130, 145), (80, 205, 215))
    ]

    for idx, (c_low, c_mid, c_high) in enumerate(water_palettes):
        w, h = 600, 600
        x = np.linspace(0, 8 * np.pi, w)
        y = np.linspace(0, 8 * np.pi, h)
        xx, yy = np.meshgrid(x, y)
        waves = np.sin(xx + 0.5 * np.cos(yy)) * np.cos(yy * 0.8 + 0.3 * np.sin(xx))
        waves = (waves - waves.min()) / (waves.max() - waves.min())

        bg_arr = np.zeros((h, w, 3), dtype=np.uint8)
        for c in range(3):
            ch = np.where(
                waves < 0.5,
                c_low[c] + (waves / 0.5) * (c_mid[c] - c_low[c]),
                c_mid[c] + ((waves - 0.5) / 0.5) * (c_high[c] - c_mid[c])
            )
            bg_arr[:, :, c] = np.clip(ch, 0, 255).astype(np.uint8)

        noise = np.random.normal(0, 7, (h, w, 3)).astype(np.int16)
        bg_arr = np.clip(bg_arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        bg_img = Image.fromarray(bg_arr).filter(ImageFilter.GaussianBlur(radius=1.0))
        bg_img.save(BACKGROUNDS_DIR / f"water_surface_{idx+1}.jpg")

    print("Generating procedural transparent plastic residue foregrounds...")
    items_to_create = {
        "plastic_bottle": [
            ("clear_bottle", (220, 235, 245, 190), "bottle"),
            ("blue_bottle", (100, 180, 240, 200), "bottle")
        ],
        "plastic_bag": [
            ("white_shopping_bag", (245, 245, 245, 175), "bag"),
            ("yellow_grocery_bag", (250, 230, 90, 180), "bag")
        ],
        "plastic_debris": [
            ("red_plastic_cup", (230, 45, 45, 230), "cup"),
            ("styrofoam_fragment", (240, 240, 240, 255), "fragment")
        ]
    }

    for cat_name, items in items_to_create.items():
        cat_path = FOREGROUNDS_DIR / cat_name
        for name, color, shape_type in items:
            size = (200, 200)
            img = Image.new("RGBA", size, (0, 0, 0, 0))
            draw = ImageDraw.Draw(img)

            if shape_type == "bottle":
                draw.rounded_rectangle([65, 60, 135, 180], radius=12, fill=color, outline=(255, 255, 255, 220), width=2)
                draw.rectangle([82, 35, 118, 60], fill=color, outline=(255, 255, 255, 200), width=1)
                draw.rectangle([80, 22, 120, 35], fill=(30, 120, 220, 240), outline=(255, 255, 255, 240), width=1)
                draw.line([75, 70, 75, 170], fill=(255, 255, 255, 150), width=3)
            elif shape_type == "bag":
                poly = [(50, 150), (40, 100), (60, 70), (75, 40), (88, 70),
                        (112, 70), (125, 40), (140, 70), (160, 100), (150, 155),
                        (120, 165), (80, 160)]
                draw.polygon(poly, fill=color, outline=(255, 255, 255, 180))
                draw.line([60, 70, 100, 140], fill=(255, 255, 255, 90), width=2)
                draw.line([140, 70, 95, 150], fill=(255, 255, 255, 90), width=2)
            elif shape_type == "cup":
                poly = [(55, 50), (145, 50), (130, 165), (70, 165)]
                draw.polygon(poly, fill=color, outline=(255, 255, 255, 230))
                draw.ellipse([50, 42, 150, 58], fill=(255, 255, 255, 210))
            elif shape_type == "fragment":
                poly = [(45, 70), (110, 40), (165, 85), (140, 150), (80, 170), (35, 120)]
                draw.polygon(poly, fill=color, outline=(200, 200, 200, 255))

            img = img.filter(ImageFilter.GaussianBlur(radius=0.5))
            img.save(cat_path / f"{name}.png")

    print("Demo procedural assets created successfully!")

create_demo_assets()

## ⚡ Section 4: CocoSynth Synthetic Data Generation Engine
This engine performs:
1. **Random Transformations**: 360-degree rotation, scaling, aspect ratio preservation, horizontal/vertical flipping.
2. **Aquatic Lighting & Appearance**: Random brightness/contrast jitter (direct sunlight glint vs cloud shadow) and alpha edge softening (Gaussian filtering on alpha mask).
3. **Polygon Contour Extraction**: Extracts contour points via OpenCV and simplifies with Shapely to produce clean COCO polygon lists `[[x1, y1, x2, y2, ...]]`.
4. **COCO Serialization**: Emits standard COCO JSON with `info`, `licenses`, `images`, `annotations`, and `categories`.

In [ ]:
# Import the CocoSynth engine from our standalone module
from cocosynth_engine import PlasticCocoSynthesizer

# Configure and instantiate synthesizer
synthesizer = PlasticCocoSynthesizer(
    foregrounds_dir=str(FOREGROUNDS_DIR),
    backgrounds_dir=str(BACKGROUNDS_DIR),
    output_dir=str(DATASET_DIR),
    image_size=(512, 512),
    min_objects_per_image=1,
    max_objects_per_image=4,
    scale_range=(0.15, 0.40),
    allow_occlusion=True,
    max_overlap_iou=0.30,
    water_blend_transparency=True
)

# Generate Training and Validation datasets
# (Adjust counts as desired, e.g. 100 train / 20 val for standard experiments)
NUM_TRAIN = 60
NUM_VAL = 15

synthesizer.generate_dataset(
    num_train=NUM_TRAIN,
    num_val=NUM_VAL,
    ann_file_name="coco_instances.json"
)

## 🔍 Section 5: COCO Dataset Inspection & Visualizer
Verify that the synthetic dataset contains accurate ground-truth instance annotations, polygon masks, and bounding boxes.

In [ ]:
def inspect_coco_dataset(json_path, images_dir, num_samples=3):
    """Plots synthetic images with overlay bounding boxes and polygon segmentation masks."""
    with open(json_path, "r") as f:
        coco_data = json.load(f)

    cat_map = {c["id"]: c["name"] for c in coco_data["categories"]}
    img_map = {im["id"]: im for im in coco_data["images"]}

    # Group annotations by image_id
    ann_by_img = {}
    for ann in coco_data["annotations"]:
        ann_by_img.setdefault(ann["image_id"], []).append(ann)

    sample_img_ids = random.sample(list(img_map.keys()), min(num_samples, len(img_map)))
    colors = ["#FF3366", "#33CC66", "#3399FF", "#FFCC00", "#CC33FF"]

    fig, axes = plt.subplots(1, len(sample_img_ids), figsize=(6 * len(sample_img_ids), 6))
    if len(sample_img_ids) == 1:
        axes = [axes]

    for ax, img_id in zip(axes, sample_img_ids):
        img_info = img_map[img_id]
        img_path = Path(images_dir) / img_info["file_name"]
        image = Image.open(img_path).convert("RGB")
        ax.imshow(image)
        ax.set_title(f"{img_info['file_name']} (ID: {img_id})")
        ax.axis("off")

        anns = ann_by_img.get(img_id, [])
        for ann_idx, ann in enumerate(anns):
            color = colors[ann_idx % len(colors)]
            cat_name = cat_map.get(ann["category_id"], "unknown")

            # Draw Bounding Box
            x, y, w, h = ann["bbox"]
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor="none", linestyle="--")
            ax.add_patch(rect)
            ax.text(x, max(0, y - 5), cat_name, color="white", fontsize=9, weight="bold",
                    bbox=dict(boxstyle="square,pad=0.2", facecolor=color, alpha=0.85))

            # Draw Polygons
            for seg in ann["segmentation"]:
                poly_pts = np.array(seg).reshape(-1, 2)
                polygon_patch = patches.Polygon(poly_pts, closed=True, facecolor=color, edgecolor=color, alpha=0.35, linewidth=2)
                ax.add_patch(polygon_patch)

    plt.tight_layout()
    plt.show()

train_json = DATASET_DIR / "train" / "coco_instances.json"
train_imgs = DATASET_DIR / "train" / "images"
inspect_coco_dataset(train_json, train_imgs, num_samples=3)

## 🧠 Section 6: Mask R-CNN Configuration & Dataset Adapter
We derive:
1. `PlasticWaterConfig` from `mrcnn.config.Config`: Configured for $512 \times 512$ image resolution, ResNet-50 backbone, anchor scales tuned for small/medium debris, and 1 + N classes (background + plastic categories).
2. `PlasticWaterDataset` from `mrcnn.utils.Dataset`: Parses the COCO JSON file, loads synthetic images, and reconstructs polygon segmentations into binary mask bitmaps `[height, width, instance_count]`.

In [ ]:
class PlasticWaterConfig(Config):
    """Configuration for training Mask R-CNN on synthetic plastic water datasets."""
    NAME = "plastic_water_residue"

    # Train on 1 GPU, batch size = 1 or 2 depending on VRAM
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1

    # Number of classes: 1 (background) + N categories
    NUM_CLASSES = 1 + len(CATEGORIES)

    # Standard resolution for synthetic water imagery
    IMAGE_MIN_DIM = 512
    IMAGE_MAX_DIM = 512
    IMAGE_RESIZE_MODE = "square"

    # Backbone architecture: resnet50 or resnet101
    BACKBONE = "resnet50"

    # Anchor scales tuned for floating debris (small fragments to large bottles/bags)
    RPN_ANCHOR_SCALES = (16, 32, 64, 128, 256)
    RPN_ANCHOR_RATIOS = [0.5, 1.0, 2.0]

    # Training iterations
    STEPS_PER_EPOCH = 30
    VALIDATION_STEPS = 5

    # Optimization
    LEARNING_RATE = 0.001
    LEARNING_MOMENTUM = 0.9
    WEIGHT_DECAY = 0.0001

    # Detection thresholds
    DETECTION_MIN_CONFIDENCE = 0.7
    DETECTION_NMS_THRESHOLD = 0.3
    DETECTION_MAX_INSTANCES = 50

config = PlasticWaterConfig()
config.display()

In [ ]:
class PlasticWaterDataset(utils.Dataset):
    """Adapter to load COCO format annotations into Matterport/akTwelve Mask R-CNN."""

    def load_coco(self, annotation_json: str, images_dir: str):
        with open(annotation_json, "r") as f:
            coco_json = json.load(f)

        source_name = "plastic_water"

        # 1. Register categories
        for category in coco_json["categories"]:
            self.add_class(source_name, category["id"], category["name"])

        # 2. Index annotations by image_id
        annotations_by_img = {}
        for ann in coco_json["annotations"]:
            annotations_by_img.setdefault(ann["image_id"], []).append(ann)

        # 3. Add images
        for img in coco_json["images"]:
            image_id = img["id"]
            file_name = img["file_name"]
            image_path = os.path.join(images_dir, file_name)

            self.add_image(
                source=source_name,
                image_id=image_id,
                path=image_path,
                width=img["width"],
                height=img["height"],
                annotations=annotations_by_img.get(image_id, [])
            )

    def load_mask(self, image_id):
        """Reconstructs binary instance masks [height, width, instance_count] from COCO polygons."""
        image_info = self.image_info[image_id]
        annotations = image_info.get("annotations", [])

        if not annotations:
            return super(PlasticWaterDataset, self).load_mask(image_id)

        instance_masks = []
        class_ids = []
        h, w = image_info["height"], image_info["width"]

        for ann in annotations:
            class_id = ann["category_id"]
            # Draw each polygon instance into a binary bitmap
            mask = Image.new('1', (w, h), 0)
            draw = ImageDraw.Draw(mask)
            for poly in ann["segmentation"]:
                if len(poly) >= 6:
                    draw.polygon(poly, outline=1, fill=1)

            mask_np = np.array(mask, dtype=bool)
            instance_masks.append(mask_np)
            class_ids.append(class_id)

        if instance_masks:
            masks_stack = np.stack(instance_masks, axis=-1)
            class_ids_arr = np.array(class_ids, dtype=np.int32)
            return masks_stack, class_ids_arr
        else:
            return np.empty([h, w, 0], dtype=bool), np.empty([0], dtype=np.int32)

# Instantiate and prepare datasets
dataset_train = PlasticWaterDataset()
dataset_train.load_coco(
    annotation_json=str(DATASET_DIR / "train" / "coco_instances.json"),
    images_dir=str(DATASET_DIR / "train" / "images")
)
dataset_train.prepare()

dataset_val = PlasticWaterDataset()
dataset_val.load_coco(
    annotation_json=str(DATASET_DIR / "val" / "coco_instances.json"),
    images_dir=str(DATASET_DIR / "val" / "images")
)
dataset_val.prepare()

print(f"Train dataset images: {len(dataset_train.image_ids)}, Classes: {dataset_train.class_names}")
print(f"Val dataset images:   {len(dataset_val.image_ids)}, Classes: {dataset_val.class_names}")

In [ ]:
# Preview loaded instance masks using mrcnn visualize utility
sample_image_id = random.choice(dataset_train.image_ids)
sample_image = dataset_train.load_image(sample_image_id)
sample_mask, sample_class_ids = dataset_train.load_mask(sample_image_id)
sample_bbox = utils.extract_bboxes(sample_mask)

visualize.display_instances(
    sample_image,
    sample_bbox,
    sample_mask,
    sample_class_ids,
    dataset_train.class_names,
    title="Sample Verification: Mask R-CNN Data Loader"
)

## 🚀 Section 7: Transfer Learning & Model Training
We load pretrained weights (`mask_rcnn_coco.h5`) trained on the standard 80-class COCO dataset.

Transfer learning proceeds in two stages:
- **Stage 1: Heads (`layers='heads'`)**: Train the Region Proposal Network (RPN), classification, bounding box, and mask heads for 10 epochs while keeping the ResNet feature extractor frozen.
- **Stage 2: Fine-Tuning (`layers='all'`)**: Train all layers end-to-end with a reduced learning rate for refined edge segmentation on complex water ripples.

In [ ]:
# Pretrained COCO weights path
COCO_WEIGHTS_PATH = BASE_DIR / "mask_rcnn_coco.h5"

if not COCO_WEIGHTS_PATH.exists():
    print("Downloading pretrained COCO weights (mask_rcnn_coco.h5)...")
    utils.download_trained_weights(str(COCO_WEIGHTS_PATH))
else:
    print(f"Pretrained COCO weights found: {COCO_WEIGHTS_PATH}")

# Create Mask R-CNN model in training mode
model = modellib.MaskRCNN(mode="training", config=config, model_dir=str(MODEL_DIR))

# Load weights, excluding final task-specific layers
model.load_weights(
    str(COCO_WEIGHTS_PATH),
    by_name=True,
    exclude=["mrcnn_class_logits", "mrcnn_bbox_fc", "mrcnn_bbox", "mrcnn_mask"]
)
print("Pretrained COCO weights loaded successfully!")

In [ ]:
# Stage 1: Train network heads
# (Set epochs higher, e.g. 15-20, when performing production training)
EPOCHS_HEADS = 5

print(f"Starting Stage 1: Training Heads for {EPOCHS_HEADS} epochs...")
model.train(
    dataset_train,
    dataset_val,
    learning_rate=config.LEARNING_RATE,
    epochs=EPOCHS_HEADS,
    layers='heads'
)

In [ ]:
# Stage 2: Fine-tune all layers (optional)
EPOCHS_ALL = 8

print(f"Starting Stage 2: Fine-tuning all layers up to epoch {EPOCHS_ALL}...")
model.train(
    dataset_train,
    dataset_val,
    learning_rate=config.LEARNING_RATE / 10.0,
    epochs=EPOCHS_ALL,
    layers='all'
)

In [ ]:
# Plot training loss curves
history = model.keras_model.history.history
if history:
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    if 'loss' in history: plt.plot(history['loss'], label='Train Total Loss', color='blue')
    if 'val_loss' in history: plt.plot(history['val_loss'], label='Val Total Loss', color='orange')
    plt.title('Total Loss Progression')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    for key in ['mrcnn_mask_loss', 'mrcnn_class_loss', 'rpn_bbox_loss']:
        if key in history:
            plt.plot(history[key], label=key)
    plt.title('Individual Component Losses')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

## 🌊 Section 8: Real-World Inference & Water Contamination Index
We now switch the model to **inference mode**, load our trained checkpoint, and detect plastic residue on real-world test images.

### Water Contamination Metric
We also calculate the **Surface Plastic Contamination Ratio**:
$$\text{Contamination Ratio (\%)} = \frac{\sum \text{Plastic Mask Pixels}}{\text{Total Water Surface Pixels}} \times 100$$

In [ ]:
class InferenceConfig(PlasticWaterConfig):
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1

inference_config = InferenceConfig()

# Initialize model in inference mode
inference_model = modellib.MaskRCNN(mode="inference", config=inference_config, model_dir=str(MODEL_DIR))

# Find latest trained checkpoint weights
latest_checkpoint = inference_model.find_last()
if latest_checkpoint:
    print(f"Loading checkpoint weights from: {latest_checkpoint}")
    inference_model.load_weights(latest_checkpoint, by_name=True)
else:
    print("No checkpoint found in logs. Loading COCO pretrained weights for testing...")
    inference_model.load_weights(str(COCO_WEIGHTS_PATH), by_name=True)

print("Inference model ready!")

In [ ]:
def evaluate_water_image(image_path, model, dataset_classes):
    """Runs detection, segments plastic objects, and computes water contamination ratio."""
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    # Run detection
    results = model.detect([image_np], verbose=0)
    r = results[0]

    # Compute contamination index
    total_pixels = image_np.shape[0] * image_np.shape[1]
    if r['masks'].shape[-1] > 0:
        # Combine all instance masks
        combined_mask = np.any(r['masks'], axis=-1)
        plastic_pixels = np.sum(combined_mask)
        contamination_ratio = (plastic_pixels / total_pixels) * 100.0
    else:
        plastic_pixels = 0
        contamination_ratio = 0.0

    print(f"\n=== Inference Results: {Path(image_path).name} ===")
    print(f"  - Detected Plastic Objects: {len(r['class_ids'])}")
    for i in range(len(r['class_ids'])):
        cls_name = dataset_classes[r['class_ids'][i]]
        score = r['scores'][i]
        print(f"    [{i+1}] {cls_name}: confidence = {score:.2%}")
    print(f"  - Water Surface Plastic Coverage: {contamination_ratio:.2f}%")

    # Visualize detection
    visualize.display_instances(
        image_np,
        r['rois'],
        r['masks'],
        r['class_ids'],
        dataset_classes,
        r['scores'],
        title=f"Plastic Residue Detection (Coverage: {contamination_ratio:.1f}%)"
    )

# Run inference on validation image or uploaded test image
test_candidates = list((DATASET_DIR / "val" / "images").glob("*.jpg")) + list(TEST_IMAGES_DIR.glob("*"))
if test_candidates:
    sample_test_path = random.choice(test_candidates)
    evaluate_water_image(sample_test_path, inference_model, dataset_train.class_names)
else:
    print("No test images found. Upload test photos into the 'test_images/' folder.")

## 📌 Section 9: Deployment & Summary

### Exporting Trained Weights
```python
# Export final model weights to a standalone file
export_path = BASE_DIR / "plastic_water_mask_rcnn_final.h5"
inference_model.keras_model.save_weights(str(export_path))
print(f"Exported weights to: {export_path}")
```

### Real-World Best Practices
1. **Diverse Backgrounds**: Ensure your background water images capture different times of day, water turbidities (crystal clear, algae blooms, sediment-heavy river water), and weather (sunny with glints vs overcast/rain ripples).
2. **Semi-Submerged Cutouts**: For transparent plastic bags and thin plastic films, adjust alpha channel opacity ($0.5-0.8$) to simulate items floating slightly below the surface.
3. **Edge Deployment**: For deployment on environmental drones or autonomous river cleaner boats, convert the Keras model weights to TensorFlow Lite (TFLite) or ONNX format for real-time edge inference.

**This project is inspired by:** [akTwelve/cocosynth](https://github.com/akTwelve/cocosynth) by Adam Kelly (Immersive Limit)